# HealthPay Agent: AI-Powered Healthcare Payment Reconciliation

**Gemma 4 Good Hackathon — Health & Sciences Track**

## Overview

US healthcare organizations lose **$262 billion annually** to administrative waste. HealthPay Agent uses **Gemma 4** to transform raw FHIR R4 claims data into actionable financial intelligence — automatically.

### Core Capabilities
| Tool | Function |
|------|----------|
| **Reconcile Claims** | Match Claims vs EOBs, identify denials/partial payments |
| **Analyze Denials** | Classify denial root causes (CARC codes), prioritize appeals |
| **A/R Reporter** | Financial Vital Signs — aging buckets, collection rates |
| **Compliance Checker** | Validate HIPAA compliance, CPT/ICD-10/HCPCS codes |
| **Risk Predictor** | Score claims by payment probability |
| **Coding Optimizer** | Identify ICD-10/CPT coding issues causing denials |

### Why Gemma 4?
- **Medical domain knowledge** — understands CARC codes, CPT/ICD-10, HIPAA
- **256K context window** — processes entire patient claim histories
- **Privacy-first** — can run locally (no PHI leaves the organization)
- **Structured reasoning** — generates actionable appeal letters, not just classifications

In [ ]:
# Install dependencies
!pip install google-genai>=1.0.0 fhir.resources>=7.0.0 httpx>=0.27.0 pydantic>=2.0.0 --quiet

In [ ]:
import os

# Configure Gemma 4 via Google AI Studio
# In Kaggle: add GEMINI_API_KEY as a secret (Settings > Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GEMINI_API_KEY = secrets.get_secret('GEMINI_API_KEY')
    print('API key loaded from Kaggle Secrets')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
    if GEMINI_API_KEY:
        print('API key loaded from environment')
    else:
        print('No API key found. Get one free at https://aistudio.google.com')

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
GEMMA4_MODEL = 'gemma-4-26b-a4b-it'
print(f'Model: {GEMMA4_MODEL}')

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

# Demo 1: Denial Root Cause Analysis
DENIAL_PROMPT = """You are a healthcare revenue cycle expert. Analyze this denied claim:

- Patient: John D., DOB 1965-03-15
- Service: CPT 99213 (Office visit, established patient, moderate complexity)
- Billed: $185.00
- Denial Code: CO-4 (The service is inconsistent with the modifier)
- Modifier used: 25 (Significant, separately identifiable E/M service)
- Payer: Blue Cross Blue Shield

Provide:
1. Root cause classification
2. Appeal priority (High/Medium/Low) with reasoning
3. Specific appeal letter talking points
4. Preventive action for future claims"""

print('Demo 1: Denial Root Cause Analysis')
print('=' * 60)
response = client.models.generate_content(
    model=GEMMA4_MODEL,
    contents=DENIAL_PROMPT,
    config=types.GenerateContentConfig(temperature=0.3, max_output_tokens=1024)
)
print(response.text)

In [ ]:
# Demo 2: A/R Aging Financial Vital Signs
ar_data = {
    'practice': 'Riverside Family Medicine',
    'total_ar': 284750.00,
    'aging_buckets': {
        '0_30_days': {'amount': 98200.00, 'claims': 142},
        '31_60_days': {'amount': 67300.00, 'claims': 89},
        '61_90_days': {'amount': 54100.00, 'claims': 71},
        '91_120_days': {'amount': 38900.00, 'claims': 52},
        'over_120_days': {'amount': 26250.00, 'claims': 38}
    },
    'top_denial_codes': [
        {'code': 'CO-4', 'desc': 'Modifier inconsistency', 'count': 23, 'amount': 4285.00},
        {'code': 'CO-97', 'desc': 'Bundled service', 'count': 18, 'amount': 3960.00},
        {'code': 'CO-16', 'desc': 'Missing information', 'count': 15, 'amount': 2875.00}
    ],
    'collection_rate_30d': 0.847,
    'avg_days_to_payment': 38.2
}

import json
AR_PROMPT = f"""You are a healthcare CFO advisor. Analyze this A/R aging report and provide:
1. Financial health score (0-100) with explanation
2. Top 3 immediate actions to improve cash flow
3. Denial pattern insights and root cause hypothesis
4. 30-day recovery forecast

A/R Data: {json.dumps(ar_data, indent=2)}"""

print('Demo 2: A/R Aging Financial Vital Signs')
print('=' * 60)
response = client.models.generate_content(
    model=GEMMA4_MODEL,
    contents=AR_PROMPT,
    config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=1024)
)
print(response.text)

In [ ]:
# Demo 3: Coding Optimization — ICD-10/CPT Issue Detection
CODING_PROMPT = """You are a medical coding expert. Review these claim codes for issues that could cause denials:

Claim #2026-0514:
- Primary Diagnosis: J06.9 (Acute upper respiratory infection, unspecified)
- Secondary Diagnosis: Z87.891 (Personal history of nicotine dependence)
- Procedure: CPT 99214 (Office visit, high complexity)
- Procedure: CPT 99406 (Smoking cessation counseling, 3-10 min)
- Modifier: None
- Payer: Medicare

Identify:
1. Coding issues or mismatches
2. Missing modifiers
3. Specificity improvements for J06.9
4. Bundling risks (NCCI edits)
5. Corrected code set recommendation"""

print('Demo 3: Coding Optimization')
print('=' * 60)
response = client.models.generate_content(
    model=GEMMA4_MODEL,
    contents=CODING_PROMPT,
    config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=1024)
)
print(response.text)

print('\n' + '=' * 60)
print('HealthPay Agent Demo Complete')
print('GitHub: https://github.com/[YOUR_USERNAME]/healthpay-agent')
print('Built with Gemma 4 + FHIR R4 + MCP Protocol')